In [57]:
import pandas as pd

# Load dataset
df = pd.read_csv("combined_pc_hardware_cleaned.csv")

df.head()

,Unnamed: 0,CPU,category,GPU,motherBoard,Ram,SSD,PowerSupply,cabinates,price
0,0.0,amd Ryzen 5 3600 with Wraith Stealth Cooler (1...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,9800.0
1,1.0,amd Ryzen 9 5900X 3.7 GHz Upto 4.8 GHz AM4 Soc...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,34890.0
2,2.0,processsor Ultra 3.5 GHz LGA 1150 Intel Core i...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,760.0
3,3.0,GIGASTAR 3.4 GHz LGA 1155 Intel i5-3570K For H...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,1890.0
4,4.0,Intel i5-12400F 4.4 GHz Upto 4.4 GHz LGA1700 S...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,13008.0


In [58]:
# Remove unwanted column if exists
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Create product name column
df["product_name"] = (
    df["CPU"]
    .fillna(df["GPU"])
    .fillna(df["Ram"])
    .fillna(df["SSD"])
    .fillna(df["PowerSupply"])
    .fillna(df["cabinates"])
)

# Remove rows without hardware name
df = df.dropna(subset=["product_name"])

# Reset index
df = df.reset_index(drop=True)

In [59]:
df.isnull().sum()

CPU              8920
category            0
GPU              9760
motherBoard     10360
Ram              8920
SSD              7720
PowerSupply      9240
cabinates        7240
price               3
product_name        0
dtype: int64

In [60]:
from sklearn.preprocessing import LabelEncoder

# Fill missing categories
df["category"] = df["category"].fillna("Unknown")

encoder = LabelEncoder()

df["category_encoded"] = encoder.fit_transform(df["category"])

In [61]:
features = ["price", "category_encoded"]

X = df[features]

y = df["category_encoded"]

In [62]:
print(df.columns)

Index(['CPU', 'category', 'GPU', 'motherBoard', 'Ram', 'SSD', 'PowerSupply',
       'cabinates', 'price', 'product_name', 'category_encoded'],
      dtype='object')


In [63]:
import plotly.express as px
fig = px.histogram(
    df,
    x="category",
    title="Hardware Category Distribution",
    color="category"
)

fig.show()

In [64]:
fig = px.histogram(
    df,
    x="price",
    nbins=50,
    title="Price Distribution of Hardware Components"
)

fig.show()

In [65]:
top_expensive = df.sort_values(by="price", ascending=False).head(10)

fig = px.bar(
    top_expensive,
    x="product_name",
    y="price",
    title="Top 10 Most Expensive Hardware",
)

fig.show()

In [66]:
from sklearn.preprocessing import LabelEncoder

# check category column
print(df["category"].unique())

# encode category
encoder = LabelEncoder()
df["category_encoded"] = encoder.fit_transform(df["category"])

# verify column exists
df.head()

['CPU' 'GPU' 'RAM' 'StorageSSD' 'PowerSupply' 'cabinates']


,CPU,category,GPU,motherBoard,Ram,SSD,PowerSupply,cabinates,price,product_name,category_encoded
0,amd Ryzen 5 3600 with Wraith Stealth Cooler (1...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,9800.0,amd Ryzen 5 3600 with Wraith Stealth Cooler (1...,0
1,amd Ryzen 9 5900X 3.7 GHz Upto 4.8 GHz AM4 Soc...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,34890.0,amd Ryzen 9 5900X 3.7 GHz Upto 4.8 GHz AM4 Soc...,0
2,processsor Ultra 3.5 GHz LGA 1150 Intel Core i...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,760.0,processsor Ultra 3.5 GHz LGA 1150 Intel Core i...,0
3,GIGASTAR 3.4 GHz LGA 1155 Intel i5-3570K For H...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,1890.0,GIGASTAR 3.4 GHz LGA 1155 Intel i5-3570K For H...,0
4,Intel i5-12400F 4.4 GHz Upto 4.4 GHz LGA1700 S...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,13008.0,Intel i5-12400F 4.4 GHz Upto 4.4 GHz LGA1700 S...,0


In [67]:
features = ["price", "category_encoded"]

X = df[features]

X.head()

,price,category_encoded
0,9800.0,0
1,34890.0,0
2,760.0,0
3,1890.0,0
4,13008.0,0


In [68]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [69]:
# -----------------------------
# Feature Engineering
# -----------------------------

from sklearn.preprocessing import LabelEncoder

# fill missing categories
df["category"] = df["category"].fillna("Unknown")

# encode category
encoder = LabelEncoder()
df["category_encoded"] = encoder.fit_transform(df["category"])

# fill missing prices
df["price"] = df["price"].fillna(df["price"].median())

# -----------------------------
# Feature Selection
# -----------------------------

features = ["price", "category_encoded"]

X = df[features]

# check missing values
print("Missing values:\n", X.isnull().sum())

# -----------------------------
# Feature Scaling
# -----------------------------

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# confirm no NaN values
print("Any NaN after scaling:", np.isnan(X_scaled).any())

Missing values:
 price               0
category_encoded    0
dtype: int64


NameError: name 'np' is not defined

In [47]:
# Define features and target

features = ["price", "category_encoded"]

X = df[features]

y = df["category_encoded"]

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_scaled)

print(similarity_matrix.shape)

(10360, 10360)


In [52]:
knn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

knn.fit(X_scaled)

NameError: name 'NearestNeighbors' is not defined

In [53]:
def recommend(name):

    index = df[df["product_name"] == name].index[0]

    X = df[features]

    X_scaled = scaler.transform(X)

    distances, indices = knn.kneighbors([X_scaled[index]])

    results = []

    for i in indices[0]:

        if i != index:  # skip the same item
            results.append(df.iloc[i][["product_name","category","price"]])

    results = pd.DataFrame(results)

    return results

In [54]:
recommend_hardware(10)

NameError: name 'recommend_hardware' is not defined

In [55]:
# Evaluate recommendation quality using distance

distances, indices = knn.kneighbors(X_scaled)

# Average similarity distance
avg_distance = np.mean(distances)

print("Average Neighbor Distance:", avg_distance)

NameError: name 'knn' is not defined

In [56]:
# Dummy evaluation using category prediction

pred = df["category_encoded"]

precision = precision_score(
    df["category_encoded"],
    pred,
    average="weighted"
)

recall = recall_score(
    df["category_encoded"],
    pred,
    average="weighted"
)

print("Precision:", precision)
print("Recall:", recall)

NameError: name 'precision_score' is not defined

In [ ]:
index = 10

distances, indices = knn.kneighbors([X_scaled[index]])

rec_df = df.iloc[indices[0]]

fig = px.bar(
    rec_df,
    x="product_name",
    y="price",
    color="category",
    title="Recommended Hardware Price Comparison"
)

fig.show()

In [ ]:
# -----------------------------
# Export Models using Joblib
# -----------------------------

import joblib
import os

print("Saving models to:", os.getcwd())

# Save KNN recommendation model
joblib.dump(knn, "knn_recommender_model.pkl")

# Save scaler
joblib.dump(scaler, "feature_scaler.pkl")

# Save label encoder
joblib.dump(encoder, "category_encoder.pkl")

# Save feature list
joblib.dump(features, "model_features.pkl")

# Optional: save processed dataframe for inference
joblib.dump(df, "hardware_dataset.pkl")

print("✅ All models exported successfully!")

In [37]:
# Remove duplicate hardware
df = df.drop_duplicates(subset=["product_name"])

# Reset index
df = df.reset_index(drop=True)

In [ ]:
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

knn = NearestNeighbors(n_neighbors=6, metric="cosine")
knn.fit(X_scaled)